# Scorer baselines: reward models and VLM judges

Every edit of the test split (2,041 samples, 5,092 edits, 4,519 pairs) scored from (original, simplified
instruction, edit) by four models, each in its own Python environment (`python -m src.scorer rewards`):

| model | what | time on 2 x T4 (to be measured by SMOKE) |
|---|---|---|
| `imagereward` | ImageReward (BLIP), a text-to-image reward model; reads the text and the edit only | minutes |
| `editreward` | EditReward-Qwen2.5-VL-7B, a reward model for edits (ICLR 2026) | about 1-2 h |
| `qwen-judge` | Qwen2.5-VL-7B-Instruct zero-shot, VIEScore prompts as in EditScore | about 15-20 h |
| `editscore` | the same with the EditScore-7B LoRA (ICLR 2026) | about 15-20 h |

Every batch is saved at once, and the run stops by itself after `HOURS` (Kaggle ends a session at 12 h, and a
session cut off keeps no output). The 7B models are spread over both GPUs. Changes from the upstream code
(float16, greedy decoding, batching, image size) are listed in `../docs/scorer.md`.

How to run, and go on in a new session:
1. Input: the dataset `cryandrrich/editjudge`. Settings: GPU T4 x2, internet on. Push the code first.
2. First run with `SMOKE = True` (8 edits per model, about 30-45 min, mostly downloading about 35 GB of
   weights): it checks every environment and prints the measured seconds per edit and the projected hours.
3. Then `SMOKE = False`, **Save Version > Save & Run All (Commit)**.
4. Next session: **Add Input > Your Work > this notebook's latest version** (its output), then Save & Run All
   again. The setup cell copies the earlier scores, and every model skips the edits already scored. Repeat
   until the summary says `"complete": true`.
5. Download `out/scorer_baselines.zip` and put its `scorer/baselines/` into `work/scorer/baselines/`, then run
   `python -m src.scorer report` (or add this notebook's output to `scorer_train.ipynb`).

In [ ]:
import glob, json, os, shutil, subprocess, sys

REPO = "https://github.com/CryAndRRich/edit-judge.git"
BRANCH = "curr"
CODE = "/kaggle/working/edit-judge"
OUT = "/kaggle/working/out"

if os.path.exists(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

def find_dataset():
    """The input folder that holds pairs.jsonl and images/ (the final dataset)."""
    for pattern in ("/kaggle/input/*/pairs.jsonl", "/kaggle/input/*/*/pairs.jsonl", "/kaggle/input/*/*/*/pairs.jsonl"):
        found = sorted(glob.glob(pattern))
        if found:
            return os.path.dirname(found[0])
    raise SystemExit("no input with pairs.jsonl: add the dataset cryandrrich/editjudge")

def find_input_folders(tail, max_depth=6):
    """Folders under /kaggle/input whose path ends with `tail`. Kaggle mounts an input at a varying depth (e.g.
    /kaggle/input/notebooks/<owner>/<notebook>/out/...), so the search walks down, but never into a dataset's
    images/ folder (thousands of folders on a slow mount) and never below `max_depth`."""
    found = []
    for root, dirs, _ in os.walk("/kaggle/input"):
        if root.endswith("/" + tail):
            found.append(root)
            dirs[:] = []
        elif root.count("/") - 2 >= max_depth:
            dirs[:] = []
        else:
            dirs[:] = [name for name in dirs if name != "images"]
    return found

def find_previous(subfolder, pattern):
    """The earlier output folder out/scorer/<subfolder> (a version of this notebook added as input) with the most
    files matching `pattern`, or None."""
    best, best_count = None, 0
    for folder in find_input_folders(f"scorer/{subfolder}"):
        count = len(glob.glob(f"{folder}/{pattern}", recursive=True))
        if count > best_count:
            best, best_count = folder, count
    return best

DATA = find_dataset()
pair_lines = [json.loads(line) for line in open(f"{DATA}/pairs.jsonl")]
print("dataset:", DATA, "-", len(pair_lines), "pairs,", sum(bool(line.get("instruction")) for line in pair_lines),
      "with an instruction")
if not any(line.get("instruction") for line in pair_lines):
    raise SystemExit("this dataset version has no instructions: add the latest version of cryandrrich/editjudge")
os.environ["EJ_DATA_DIR"] = DATA           # read-only input
os.environ["EJ_OUTPUT_DIR"] = OUT          # all results are written here
os.environ["EJ_ENVS_DIR"] = "/tmp/ej-envs"  # one Python environment per group of conflicting packages
os.makedirs(f"{OUT}/scorer", exist_ok=True)
os.chdir(CODE)

previous = find_previous("baselines", "*.jsonl")
if previous:
    shutil.copytree(previous, f"{OUT}/scorer/baselines", dirs_exist_ok=True)
    print("went on from", previous)
else:
    print("no earlier scores found: starting from the beginning")

# Model weights go to /tmp: /kaggle/working is limited to 20 GB and is saved as the notebook's output
os.environ["HF_HOME"] = "/tmp/hf"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
try:                                       # optional Kaggle secret HF_TOKEN (Add-ons > Secrets): faster downloads
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    os.environ["HF_HUB_VERBOSITY"] = "error"

In [ ]:
!pip -q install uv                        # fast creation of the per-model environments (src/common/envs.py)
!nvidia-smi --query-gpu=name,memory.total --format=csv

def run_scorer(*args):
    """python -m src.scorer <args>: prints every line as it comes, and returns the output."""
    process = subprocess.Popen([sys.executable, "-u", "-m", "src.scorer", *map(str, args)],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, args)
    return "".join(lines)

run_scorer("list")

In [ ]:
MODELS = ["imagereward", "editreward", "qwen-judge", "editscore"]     # fast models first
HOURS = 10              # stop before the next batch after this many hours, all models together
SMOKE = False           # True: 8 edits per model, to check the environments and measure the speed

In [ ]:
output = run_scorer("rewards", "--models", ",".join(MODELS), "--max-hours", HOURS, *(["--limit", 8] if SMOKE else []))

In [ ]:
summary = json.load(open(f"{OUT}/scorer/baselines/summary.json"))
for name, status in summary["models"].items():
    meta_path = f"{OUT}/scorer/baselines/{name}.meta.json"
    per_edit = json.load(open(meta_path))["seconds_per_edit"] if os.path.exists(meta_path) else None
    hours = round(status["left"] * per_edit / 3600, 1) if per_edit else None
    failed = "" if status["exit_code"] in (0, None) else f" - FAILED ({status['exit_code']}), see the log above"
    print(f"{name}: {status['scored']} scored, {status['left']} left, {per_edit} s/edit, about {hours} h left{failed}")
print("complete" if summary["complete"] else "not complete: add this version's output as input and run again")

import zipfile
keep = glob.glob(f"{OUT}/scorer/baselines/*") + glob.glob(f"{OUT}/scorer/logs/*")
with zipfile.ZipFile(f"{OUT}/scorer_baselines.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in keep:
        if os.path.isfile(path):
            archive.write(path, os.path.relpath(path, OUT))
print(f"{OUT}/scorer_baselines.zip:", len(keep), "files")